**Benchmarking Script**

Warmup steps: 5 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.9 ± 0.2 | 295.4 ± 0.1 | 313.4 ± 0.2 |
| medium | 291.8 ± 0.2 | 870.4 ± 0.4 | 935.5 ± 1.6 |
| large | 617.1 ± 3.3 | 1893.9 ± 29.9 | OOM/ERR |

The standard deviations (see above) are small. Decided not to run the XL and 10B models since they likely will not run on laptop.

Now I ran with 0 warm up steps (my laptop crashed with large model, so I omitted this):

Warmup steps: 0 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 108.6 ± 26.2 | 300.6 ± 12.1 | 319.1 ± 11.2 |
| medium | 296.3 ± 10.0 | 877.0 ± 12.6 | 943.6 ± 15.1 |

As can be seen the times are slightly longer, likely due to cold cache.

And finally running with one warmup step:

Warmup steps: 1 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.7 ± 0.1 | 294.8 ± 0.1 | 313.1 ± 0.5 |
| medium | 291.5 ± 0.4 | 867.5 ± 0.1 | 933.7 ± 1.0 |

Even just one warmup step seems to make a difference.

**PyTorch Profiling**

Note: I am not using Nsys but rather torch.profile for compatibility with my macbook.

(a) Total time spent for forward is:

| Model | Batch Size | Time (ms) |
|---|---|---|
| small | 256 | 53.80 |
| small | 512 | 115.05 |
| medium | 256 | 145.75 |
| medium | 512 | 319.51 |

For batch size 512, the times without profiling for small (99.9ms) and medium (291.8ms) are slightly smaller as expected.

(b) The `aten::bmm` (batched matrix multiply) kernel takes most time on the forwards pass, consuming ~30% of runtime regardless of model (small/medium). This kernel is called 217 times (medium model) and 109 times (small model). While the forwards pass is dominated by `bmm`, the backwards pass also has `aten::sum` as a significant component, which makes sense since the autograd engine performs lots of summations. (*weird behaviour seen with medium model and 1024 context length: `sum` becomes negligible and `bmm` jumps to ~80%*).

(c) `einsum`, `mean`, `permute`, `max`, `mul`

(d) Compared to inference, when performing a full (forwards/backwards/optimizer) pass, the fraction of time spent on `bmm` falls slightly and time spent on `addcdiv` and `addcmul` rises slightly (which are elementwise operations directly related to steps within the AdamW algorithm).

(e) To perform this comparison, markers were added within the `scaled_dot_product_attention` function. The FLOPs was estimated using analytical formulas. Although the FLOPs ratio (mul FLOPs / softmax FLOPs) is ~85, the runtime ratio is just ~1.3x suggesting softmax is memory bound and that it takes a long time despite relatively little computation.

**NSys Profiling on AWS Cluster**

(a) (skipped)

(b) The `xmma_gemm` kernels take most time on forwards pass (matrix multiply kernels). Using the medium model for reference:

| Context length | Kernel | Runtime % | No. calls |
| -- | -- | -- | -- |
| 256 | sm90_xmma_gemm...tn...64x128x32 | 23.2 | 120 |
| 512 | sm90_xmma_gemm...tn...128x128x32 | 25.2 | 169 |
| 1024 | sm90_xmma_gemm...tn...128x128x32 | 16.2 | 168 |

Note: for 256, there are also 48 calls for `sm90_xmma_gemm...tn...128x256x32` (which added to the 120 calls shown above, gives 168 calls). I therefore think the same number of kernel calls are happening, but depending upon context length the exact kernels used differ.

Also note: there are different kernel transpose variants (`nn`, `tn`, `nt`) which complicate matters. In forwards pass the `tn` dominates.

For 512-context medium, in backwards pass, no single `xma_gemm` kernel takes a large (>10%) runtime, although summing the transpose variants shows matrix multiplication consumes ~33% of runtime in both forwards and backwards passes. Therefore the matmul kernels are more varied in backwards pass, likely a direct consequence of needing gradients with respect to both inputs and weights. The `MulFunctor` kernel jumps from 1.620ms (forward) to 7.387ms (backward), a 4.6× increase, because the backward pass adds gradient computations for SwiGLU, RoPE, and residual connections, all of which involve elementwise multiplications.

(c) Numbers for medium model: `vectorized_elementwise_kernel BinaryFunctor` (elementwise multiply), 6.1% at `ctx=256`, 7.1% at `ctx=512`, 5.5% at `ctx=1024`; `reduce_kernel MaxOps`, finds max for numerical stability (3.3% to 4.8%); `exp_kernel`, computes exponentials (2.2% to 7.4%)

(d) Using 512-context medium, the fraction of time spent on matrix multiplication decreases slightly from ~33% to ~29%. In addition, new kernels such as `addcdiv_cuda_kernel` and `addcmul_cuda_kernel` appear.

(e) (skipped)

**Mixed Precision Accumulation**

FP32 gives the most accurate answer. FP16 gives ~5% error. The third and fourth runs give same answer revealing FP16 promotes to FP32. This is slightly worse than pure FP32 because the FP16 error manifests before the promotion occurs.

**Benchmarking Mixed Precision**

| Component | dtype | Reason |
|---|---|---|
| Model parameters | float32 | autocast never changes stored parameter dtypes |
| fc1 output | float16 | matmul is on autocast whitelist |
| LayerNorm output | float32 | reduction op, excluded from whitelist for numerical stability |
| logits (fc2 output) | float16 | matmul is on autocast whitelist |
| loss | float32 | reduction op, excluded from whitelist for numerical stability |
| gradients | float32 | always float32 regardless of autocast |
 
---
 
**EXPLANATION**
 
**Model parameters** remain float32 throughout — autocast only affects computation dtype, not storage dtype. The float32 weights are temporarily cast to float16 for eligible operations but the stored values are unchanged.
 
**fc1 output is float16** because `nn.Linear` (matmul) is on PyTorch's autocast whitelist of eligible operations. The float32 weights are cast to float16 for the computation and the output is float16.
 
**LayerNorm output is float32** because LayerNorm involves reductions (mean, variance) that are numerically sensitive and prone to overflow/underflow in float16. PyTorch's autocast whitelist explicitly excludes LayerNorm for this reason. This requires a dtype cast from float16 → float32, which has a small but measurable cost (visible as `direct_copy_kernel` in nsys profiles).
 
**Logits (fc2 output) are float16** — fc2 is another matmul on the whitelist. Even though its input from LayerNorm is float32, autocast casts it back to float16 for the matmul, giving float16 output.
 
**Loss is float32** — CrossEntropyLoss involves reductions (sum, log, exp) kept in float32 for numerical stability, even though the input logits are float16.
 
**Gradients are float32** — gradients are always computed and stored in float32 regardless of autocast. This is critical for training stability — float16 gradients could underflow to zero for small gradient values, causing parameters to stop updating.
 
---
 
**FP16 VS BF16**
 
If using `dtype=torch.bfloat16` instead of `torch.float16`, LayerNorm and other reductions can safely run in bfloat16 because bfloat16 has the same dynamic range as float32 (8 exponent bits vs 5 for float16). This reduces the number of dtype conversions needed, improving performance at the cost of slightly lower mantissa precision (7 bits vs 10 for float16).

**Benchmarking Mixed Precision**

Running BF16 mixed precision against FP32 across small, medium, and large model sizes, we observed modest speedups of 10-20% rather than the theoretical ~2× improvement. 

This is explained by three factors: 

- the H100 already uses TF32 tensor cores for FP32 matmuls by default (visible in nsys kernel names as `f32f32_tf32f32_f32`), meaning the FP32 baseline is already faster than true scalar FP32
- only ~33% of forward pass kernel time is spent on matmuls with the remainder dominated by softmax and elementwise operations
- Amdahl's Law limits the overall speedup to ~1.20× even with a 2× matmul speedup. 

Speedups were consistently larger for forward+backward passes than forward-only, and grew modestly with model size, consistent with larger models being more compute-bound and better utilising tensor cores.

**Memory Profiling**

![](images/memory_prof_xl_fwd_512.png)
Memory profile of XL forward pass for 512 context length.

![](images/memory_prof_xl_full_512.png)
Memory profile of XL full pass for 512 context length.

(b) 

|Run|Peak Memory Usage (GB)|
|--|--|
|XL `ctx=128` fwd|12.8|
|XL `ctx=512` fwd|13.3|
|XL `ctx=2048` fwd|21.1|
|XL `ctx=128` full|50|
|XL `ctx=512` full|65|
|XL `ctx=2048` full|OOM|

Note the XL model has ~3.4B params. At FP32, this means ~13GB storage. All forward profiles have this baseline consumption. The forward profiles have 32 spikes for the calculation of the 32 layers. These spikes are small (~125MB) for `ctx=128`, larger (~1GB) for `ctx=512` and large (~7.5GB) for `ctx=2048`. This makes sense since greater context length makes no change to model parameters but will change the working memory needed to compute through each layer.

The full profile is triangular. From warmup steps I have already allocated memory for the first/second moments of each param. This means each param needs three values, so we expect ~39GB persistent allocation. Both 128 and 512 profiles have ~38GB of continuous storage used as expected; 2048 gives OOM. The activations are stored when performing the forward pass hence the triangular build-up. These are needed for the backwards pass. When starting backwards pass, the activations can start to be deallocated from the model back-end, so the profile starts to ramp down. More allocations are gradually added, which are the gradient tensors. These need to be kept in memory for the optimization step. The optimization step itself does not need any more memory but does takes a little time, hence things appear to stretch horizontally at the end.

(c) Mixed precision does not significantly affect memory usage. Note `cached_enabled` should be set to `False` otherwise substantial memory (~6GB from tests) can be allocated for caching of the BF16 values.

(d) The residual stream tensor has shape `batch` X `seq_len` X `d_model`. This gives (4 X 512 X 2560) x 4 bytes equally ~20MB.

(e) The visuals are slightly hard to interpret but looking at the stack trace the largest allocations (for 2048 `ctx`) seem to be ~2GB in size. The `scaled_dot_product_attention` function is in the stack trace. The attention score matrix has dimensions `batch` X `heads` X `seq` X `seq` so for 2048 this has 4 X 32 X 2048 X 2048 parameters and ~2GB size.

(f) I am struggling with this a little so am skipping this part for the time being.

**Memory Optimal Gradient Checkpointing**

(a) Having each transformer block checkpointed minimises peak activation memory. The asymptotic peak activation memory is O(1) since we only ever store one block's worth of activations in memory. The compute would increase by 33% (assuming that a forwards pass is roughly half the computational cost of a backwards pass), since we go from F + B to 2F + B where F and B are forwards and backwards passes respectively. Note to further reduce peak activation memory, I could have nested checkpoints where the two sublayers (attention and feedforward) of the transformer block are also checkpointed.

(b) The best strategy is now indeed using one checkpoint per transformer block. For XL full pass, using `ctx=512` with this strategy reduces peak memory usage from ~65GB to ~51.5GB. Also, `ctx=2048`, which previously gave OOM for full pass, now runs with peak memory ~63.5GB. The peak memory saving associated with going from checkpointing every block individually to checkpointing every two blocks is negligible (as verified with a test using `ctx=512`).

**PyTorch Attention Benchmarking**

Batch size: 8 | Single head (no multi-head) | Warmup: 5 steps | Measurement: 100 steps | Device: CUDA

| d_k | seq_len | Forward (ms) | Backward (ms) | Mem before bwd (MB) |
|---|---|---|---|---|
| 16 | 256 | 0.31 ± 0.02 | 0.68 ± 0.02 | 68.6 |
| 16 | 1024 | 0.32 ± 0.02 | 0.73 ± 0.04 | 132.1 |
| 16 | 4096 | 3.34 ± 0.04 | 6.85 ± 0.04 | 1128.4 |
| 16 | 8192 | 12.99 ± 0.01 | 26.25 ± 0.02 | 4304.8 |
| 16 | 16384 | 51.20 ± 0.02 | 104.58 ± 0.07 | 16993.5 |
| 32 | 256 | 0.30 ± 0.01 | 0.60 ± 0.01 | 69.1 |
| 32 | 1024 | 0.32 ± 0.01 | 0.74 ± 0.01 | 134.1 |
| 32 | 4096 | 3.43 ± 0.14 | 6.87 ± 0.01 | 1136.4 |
| 32 | 8192 | 13.06 ± 0.01 | 26.32 ± 0.02 | 4320.8 |
| 32 | 16384 | 51.25 ± 0.02 | 104.50 ± 0.05 | 17025.5 |
| 64 | 256 | 0.31 ± 0.01 | 0.65 ± 0.04 | 70.1 |
| 64 | 1024 | 0.33 ± 0.01 | 0.75 ± 0.01 | 138.1 |
| 64 | 4096 | 3.41 ± 0.06 | 6.95 ± 0.07 | 1152.4 |
| 64 | 8192 | 13.07 ± 0.04 | 26.48 ± 0.12 | 4352.8 |
| 64 | 16384 | 51.54 ± 0.13 | 104.90 ± 0.29 | 17089.5 |
| 128 | 256 | 0.32 ± 0.01 | 0.69 ± 0.03 | 72.1 |
| 128 | 1024 | 0.35 ± 0.01 | 0.81 ± 0.04 | 146.1 |
| 128 | 4096 | 3.44 ± 0.06 | 7.01 ± 0.03 | 1184.4 |
| 128 | 8192 | 13.23 ± 0.01 | 26.81 ± 0.23 | 4416.8 |
| 128 | 16384 | 52.12 ± 0.06 | 105.67 ± 0.12 | 17217.5 |

Attention Memory Accounting:

Since no configurations OOM on the H100, we analyse the largest configuration (d_k=128, seq_len=16384, batch=8) as the most memory-intensive case.

| Component | Size |
|---|---|
| Q, K, V inputs (3 × batch × seq × d_k × 4 bytes) | ~192 MB |
| Attention scores (batch × seq × seq × 4 bytes) | ~8 GB |
| Softmax output (batch × seq × seq × 4 bytes) | ~8 GB |
| Output tensor (batch × seq × d_k × 4 bytes) | ~64 MB |
| **Total** | **~16.3 GB** |

This matches the observed 16,993.5 MB (~16.6 GB), with the small discrepancy from PyTorch allocator overhead.

Memory before backward scales as O(T²), dominated by the attention score and softmax matrices:

| seq_len | Mem before bwd (MB) | Ratio to previous |
|---|---|---|
| 256 | 72.1 | — |
| 1024 | 146.1 | 2.0× |
| 4096 | 1184.4 | 8.1× |
| 8192 | 4416.8 | 3.7× |
| 16384 | 17217.5 | 3.9× |

Ratios approach 4× for each 2× increase in seq_len, consistent with quadratic T² scaling. The dominant memory term is:

$$\text{mem} \approx 2 \times \text{batch} \times T^2 \times 4\text{ bytes} = O(T^2)$$

FlashAttention eliminates the O(T²) memory cost by never materialising the full attention matrix. Instead of saving the `(batch, seq, seq)` scores and softmax output for the backward pass, it tiles the computation into blocks, recomputing attention scores on the fly during the backward pass from Q, K, V.

**Torch Compile**

(a) The `compile` command does not produce meaningful changes in runtime. This is likely because the large (`batch_size` X `seq_len` X `seq_len`) matrix makes the attention algorithm memory bound.

(b) For medium model with batch size of 8 and sequence length of 512:

| Mode | Uncompiled (ms) | Compiled (ms) | Speedup |
|---|---|---|---|
| Forward | 46 | 26 | 1.77x |
| Full | 173 | 116 | 1.49x |

**Flash Attention Benchmarking**
 
**GPU:** NVIDIA H100 80GB HBM3 | **Batch size:** 1 | **Causal:** True | **Times in ms**
 
## BFloat16
 
| seq_len | d_head | fwd_triton | fwd_pytorch | bwd_triton | bwd_pytorch | e2e_triton | e2e_pytorch | speedup_fwd | speedup_e2e |
|---|---|---|---|---|---|---|---|---|---|
| 128 | 16 | 0.025 | 0.249 | 0.423 | 0.498 | 0.513 | 0.795 | 9.98x | 1.55x |
| 128 | 32 | 0.025 | 0.250 | 0.462 | 0.501 | 0.548 | 0.963 | 10.03x | 1.76x |
| 128 | 64 | 0.028 | 0.256 | 0.478 | 0.514 | 0.619 | 1.010 | 9.28x | 1.63x |
| 128 | 128 | 0.026 | 0.254 | 0.447 | 0.518 | 0.561 | 1.015 | 9.79x | 1.81x |
| 256 | 16 | 0.024 | 0.260 | 0.516 | 0.536 | 0.620 | 1.040 | 10.66x | 1.68x |
| 256 | 32 | 0.031 | 0.260 | 0.500 | 0.514 | 0.615 | 0.869 | 8.32x | 1.41x |
| 256 | 64 | 0.033 | 0.267 | 0.428 | 0.461 | 0.500 | 0.845 | 8.08x | 1.69x |
| 256 | 128 | 0.034 | 0.265 | 0.426 | 0.489 | 0.496 | 0.870 | 7.79x | 1.75x |
| 512 | 16 | 0.028 | 0.266 | 0.435 | 0.467 | 0.494 | 0.851 | 9.48x | 1.72x |
| 512 | 32 | 0.031 | 0.264 | 0.453 | 0.464 | 0.518 | 0.850 | 8.61x | 1.64x |
| 512 | 64 | 0.031 | 0.262 | 0.434 | 0.467 | 0.496 | 0.850 | 8.56x | 1.71x |
| 512 | 128 | 0.028 | 0.263 | 0.425 | 0.464 | 0.488 | 0.849 | 9.34x | 1.74x |
| 1024 | 16 | 0.029 | 0.264 | 0.423 | 0.463 | 0.489 | 0.849 | 9.13x | 1.74x |
| 1024 | 32 | 0.027 | 0.263 | 0.426 | 0.467 | 0.495 | 0.849 | 9.70x | 1.72x |
| 1024 | 64 | 0.032 | 0.264 | 0.429 | 0.463 | 0.493 | 0.849 | 8.33x | 1.72x |
| 1024 | 128 | 0.030 | 0.250 | 0.430 | 0.463 | 0.491 | 0.854 | 8.28x | 1.74x |
| 2048 | 16 | 0.027 | 0.266 | 0.430 | 0.470 | 0.493 | 0.855 | 9.73x | 1.73x |
| 2048 | 32 | 0.034 | 0.269 | 0.438 | 0.489 | 0.514 | 0.858 | 8.01x | 1.67x |
| 2048 | 64 | 0.031 | 0.264 | 0.425 | 0.458 | 0.486 | 0.844 | 8.52x | 1.74x |
| 2048 | 128 | 0.036 | 0.263 | 0.430 | 0.471 | 0.489 | 0.855 | 7.34x | 1.75x |
| 4096 | 16 | 0.038 | 0.360 | 0.431 | 0.612 | 0.500 | 0.944 | 9.37x | 1.89x |
| 4096 | 32 | 0.041 | 0.360 | 0.436 | 0.610 | 0.506 | 0.941 | 8.88x | 1.86x |
| 4096 | 64 | 0.048 | 0.360 | 0.485 | 0.614 | 0.554 | 0.945 | 7.50x | 1.70x |
| 4096 | 128 | 0.060 | 0.360 | 0.497 | 0.621 | 0.564 | 0.950 | 5.96x | 1.68x |
| 8192 | 16 | 0.070 | 1.313 | 1.003 | 2.114 | 1.059 | 3.404 | 18.88x | 3.21x |
| 8192 | 32 | 0.074 | 1.315 | 1.007 | 2.111 | 1.069 | 3.395 | 17.69x | 3.18x |
| 8192 | 64 | 0.088 | 1.313 | 1.049 | 2.113 | 1.116 | 3.395 | 14.97x | 3.04x |
| 8192 | 128 | 0.112 | 1.325 | 1.116 | 2.133 | 1.199 | 3.420 | 11.80x | 2.85x |
| 16384 | 16 | 0.184 | 4.868 | 3.716 | 7.927 | 3.865 | 12.798 | 26.45x | 3.31x |
| 16384 | 32 | 0.207 | 4.872 | 3.769 | 7.934 | 3.935 | 12.808 | 23.51x | 3.26x |
| 16384 | 64 | 0.259 | 4.885 | 3.917 | 7.958 | 4.085 | 12.833 | 18.85x | 3.14x |
| 16384 | 128 | 0.358 | 4.937 | 4.019 | 8.017 | 4.348 | 12.911 | 13.77x | 2.97x |
| 32768 | 16 | 0.687 | 19.213 | 14.532 | 31.333 | 15.188 | 50.550 | 27.95x | 3.33x |
| 32768 | 32 | 0.779 | 19.243 | 14.668 | 31.402 | 15.301 | 50.573 | 24.71x | 3.31x |
| 32768 | 64 | 0.946 | 19.196 | 15.320 | 31.371 | 16.146 | 50.605 | 20.29x | 3.13x |
| 32768 | 128 | 1.406 | 19.305 | 16.101 | 31.462 | 17.331 | 50.731 | 13.73x | 2.93x |
| 65536 | 16 | 2.631 | 76.265 | 60.180 | 125.096 | 62.768 | 201.241 | 28.98x | 3.21x |
| 65536 | 32 | 2.977 | 76.267 | 61.156 | 125.119 | 64.129 | 201.359 | 25.62x | 3.14x |
| 65536 | 64 | 3.651 | 76.526 | 62.153 | 125.350 | 66.124 | 201.831 | 20.96x | 3.05x |
| 65536 | 128 | 5.644 | 76.739 | 62.109 | 125.946 | 78.999 | 202.487 | 13.60x | 2.56x |
 
## Float32
 
| seq_len | d_head | fwd_triton | fwd_pytorch | bwd_triton | bwd_pytorch | e2e_triton | e2e_pytorch | speedup_fwd | speedup_e2e |
|---|---|---|---|---|---|---|---|---|---|
| 128 | 16 | 0.031 | 0.255 | 0.422 | 0.501 | 0.531 | 0.996 | 8.10x | 1.88x |
| 128 | 32 | 0.026 | 0.264 | 0.405 | 0.506 | 0.529 | 0.879 | 10.06x | 1.66x |
| 128 | 64 | 0.032 | 0.260 | 0.369 | 0.456 | 0.436 | 0.850 | 8.06x | 1.95x |
| 128 | 128 | 0.031 | 0.248 | 0.342 | 0.434 | 0.409 | 0.810 | 8.11x | 1.98x |
| 256 | 16 | 0.031 | 0.261 | 0.368 | 0.459 | 0.438 | 0.846 | 8.38x | 1.93x |
| 256 | 32 | 0.032 | 0.264 | 0.367 | 0.456 | 0.438 | 0.847 | 8.30x | 1.93x |
| 256 | 64 | 0.031 | 0.264 | 0.369 | 0.463 | 0.438 | 0.853 | 8.48x | 1.95x |
| 256 | 128 | 0.030 | 0.257 | 0.359 | 0.460 | 0.425 | 0.839 | 8.71x | 1.97x |
| 512 | 16 | 0.032 | 0.261 | 0.366 | 0.452 | 0.437 | 0.840 | 8.20x | 1.92x |
| 512 | 32 | 0.034 | 0.269 | 0.380 | 0.468 | 0.452 | 0.864 | 8.03x | 1.91x |
| 512 | 64 | 0.031 | 0.262 | 0.365 | 0.459 | 0.433 | 0.850 | 8.40x | 1.96x |
| 512 | 128 | 0.042 | 0.258 | 0.368 | 0.460 | 0.436 | 0.832 | 6.22x | 1.91x |
| 1024 | 16 | 0.031 | 0.258 | 0.363 | 0.459 | 0.434 | 0.835 | 8.34x | 1.93x |
| 1024 | 32 | 0.030 | 0.255 | 0.359 | 0.457 | 0.421 | 0.837 | 8.63x | 1.99x |
| 1024 | 64 | 0.038 | 0.257 | 0.355 | 0.465 | 0.429 | 0.844 | 6.69x | 1.97x |
| 1024 | 128 | 0.071 | 0.259 | 0.362 | 0.465 | 0.431 | 0.846 | 3.65x | 1.96x |
| 2048 | 16 | 0.033 | 0.259 | 0.366 | 0.462 | 0.437 | 0.844 | 7.85x | 1.93x |
| 2048 | 32 | 0.037 | 0.257 | 0.366 | 0.460 | 0.437 | 0.844 | 6.97x | 1.93x |
| 2048 | 64 | 0.067 | 0.258 | 0.365 | 0.459 | 0.439 | 0.842 | 3.87x | 1.92x |
| 2048 | 128 | 0.130 | 0.258 | 0.364 | 0.450 | 0.439 | 0.833 | 1.98x | 1.90x |
| 4096 | 16 | 0.047 | 0.530 | 0.365 | 0.973 | 0.440 | 1.476 | 11.25x | 3.35x |
| 4096 | 32 | 0.062 | 0.533 | 0.373 | 0.980 | 0.448 | 1.485 | 8.58x | 3.32x |
| 4096 | 64 | 0.125 | 0.538 | 0.414 | 0.994 | 0.489 | 1.500 | 4.29x | 3.07x |
| 4096 | 128 | 0.242 | 0.544 | 0.429 | 1.008 | 0.553 | 1.519 | 2.25x | 2.75x |
| 8192 | 16 | 0.088 | 1.916 | 0.961 | 3.417 | 1.028 | 5.309 | 21.74x | 5.16x |
| 8192 | 32 | 0.119 | 1.915 | 0.960 | 3.421 | 1.055 | 5.309 | 16.14x | 5.03x |
| 8192 | 64 | 0.244 | 1.930 | 0.996 | 3.440 | 1.216 | 5.343 | 7.92x | 4.40x |
| 8192 | 128 | 0.458 | 1.938 | 1.053 | 3.485 | 1.476 | 5.376 | 4.23x | 3.64x |
| 16384 | 16 | 0.251 | 7.348 | 3.578 | 13.174 | 3.802 | 20.521 | 29.24x | 5.40x |
| 16384 | 32 | 0.351 | 7.368 | 3.625 | 13.196 | 3.944 | 20.563 | 20.97x | 5.21x |
| 16384 | 64 | 0.775 | 7.392 | 3.702 | 13.258 | 4.440 | 20.644 | 9.54x | 4.65x |
| 16384 | 128 | 1.763 | 7.420 | 3.794 | 13.291 | 5.535 | 20.689 | 4.21x | 3.74x |
| 32768 | 16 | 0.927 | 29.071 | 13.988 | 52.188 | 14.871 | 81.244 | 31.36x | 5.46x |
| 32768 | 32 | 1.294 | 29.067 | 14.539 | 52.143 | 15.244 | 81.181 | 22.46x | 5.33x |
| 32768 | 64 | 3.047 | 29.597 | 15.247 | 52.864 | 17.727 | 82.235 | 9.71x | 4.64x |
| 32768 | 128 | 6.995 | 29.850 | 15.550 | 53.048 | 22.208 | 82.839 | 4.27x | 3.73x |
| 65536 | 16 | 3.600 | OOM | 60.470 | OOM | 63.873 | OOM | N/A | N/A |
| 65536 | 32 | 5.078 | OOM | 61.241 | OOM | 66.824 | OOM | N/A | N/A |
| 65536 | 64 | 12.087 | OOM | 62.198 | OOM | 74.822 | OOM | N/A | N/A |
| 65536 | 128 | 27.790 | OOM | 63.085 | OOM | 89.858 | OOM | N/A | N/A |

**Distributed Communication (Single Node)**

All-Reduce Benchmark Results (NCCL, Single Node)

**GPU:** NVIDIA H100 80GB HBM3 | **Backend:** NCCL | **Warmup:** 5 steps | **Measurement:** 10 steps | **Times in ms**

World Size: 2

| Size | Mean (ms) | Std (ms) | Min (ms) | Max (ms) |
|---|---|---|---|---|
| 1MB | 0.074 | 0.009 | 0.067 | 0.099 |
| 10MB | 0.091 | 0.001 | 0.089 | 0.093 |
| 100MB | 0.418 | 0.002 | 0.414 | 0.422 |
| 1GB | 3.253 | 0.025 | 3.211 | 3.298 |

World Size: 4

| Size | Mean (ms) | Std (ms) | Min (ms) | Max (ms) |
|---|---|---|---|---|
| 1MB | 0.072 | 0.005 | 0.067 | 0.083 |
| 10MB | 0.118 | 0.001 | 0.116 | 0.120 |
| 100MB | 0.526 | 0.005 | 0.520 | 0.531 |
| 1GB | 4.548 | 0.017 | 4.519 | 4.572 |

World Size: 6

| Size | Mean (ms) | Std (ms) | Min (ms) | Max (ms) |
|---|---|---|---|---|
| 1MB | 0.075 | 0.004 | 0.071 | 0.085 |
| 10MB | 0.125 | 0.004 | 0.120 | 0.134 |
| 100MB | 0.576 | 0.003 | 0.571 | 0.583 |
| 1GB | 5.034 | 0.009 | 5.022 | 5.049 |

Discussion

All-reduce time increases with both data size and world size, consistent with NCCL's ring allreduce algorithm where each GPU must transfer $2 \times \frac{N-1}{N}$ times the message size; more GPUs means more total data transferred per GPU. At small sizes (1MB) the time is nearly constant across world sizes (~0.07ms), indicating the operation is latency-bound and dominated by NVLink coordination overhead rather than data transfer.